# 지도학습 이해


---
---



## **1. 지도학습(supervised learning) 이해**
- 정답(label)이 주어진 데이터를 이용해 모델을 학습시키는 기계학습 방식
- 모델은 입력(X)과 출력(Y) 간의 관계를 학습하고, 새로운 입력에 대한 예측을 수행



---



### **[실습] 자연어 처리(NLP) 분야에서의 지도학습(분류) 예**


#### **예제1. 텍스트 분류(뉴스 주제 분류)**
- **태스크** : 뉴스 주제 분류
- **문제유형** : 다중 클래스 분류
- **데이터셋** : 20 Newsgroups (scikit_learn 내장)
- **특징추출** : TF-IDF(N-gram: bigram)
- **분류기** : Logistic regression
- **핵심내용**: 가장 기본적인 문서 분류 파이프라인

In [ ]:
# --- 1. 텍스트 분류: 20 Newsgroups ---
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, accuracy_score


# 1) 데이터 로드 (훈련/테스트 분리 제공)
cats = ['sci.space', 'rec.sport.baseball', 'comp.graphics', 'talk.politics.mideast']
train = fetch_20newsgroups(subset='train', categories=cats, remove=('headers','footers','quotes'))
test  = fetch_20newsgroups(subset='test',  categories=cats, remove=('headers','footers','quotes'))


# 2) 파이프라인: TF-IDF → 로지스틱 회귀(멀티클래스)
pipe = make_pipeline(TfidfVectorizer(min_df=3, ngram_range=(1,2)),
                     LogisticRegression(max_iter=300))


# 3) 학습
pipe.fit(train.data, train.target)


# 4) 평가
pred = pipe.predict(test.data)
print("✅ Accuracy:", accuracy_score(test.target, pred))
print(classification_report(test.target, pred, target_names=test.target_names))


# 5) 사용 예시
sample = ["SpaceX just launched another rocket to the ISS.",
          "OpenGL shaders can speed up graphics rendering.",
          "The team won the baseball world series!",
          "The negotiation in the middle east escalated."]
print("✅ sample 예측결과: ", pipe.predict(sample))          # 예측된 정수 라벨
print([test.target_names[i] for i in pipe.predict(sample)])  # 라벨 이름


#### **예제2. 감성 분류(긍/부정)**
NLTK의 movie_reviews 코퍼스(긍/부정 라벨)를 사용해 문서 분류(이진 분류)를 수행합니다.

- **태스크** : 감성 분석
- **문제유형** : 이진 분류
- **데이터셋** : nltk, movie_reviews
- **특징추출** : TF-IDF(N-gram: bigram)
- **분류기** : Linear SVM
- **핵심내용**: 리뷰 텍스트의 긍/부정 판

In [ ]:
# --- 2. 감성 분석: NLTK movie_reviews (긍/부정) ---
import nltk
nltk.download('movie_reviews')

from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, accuracy_score

# 1) 문서/라벨 구성
docs = []
labels = []
for cat in movie_reviews.categories():          # 'pos', 'neg'
    for fid in movie_reviews.fileids(cat):
        docs.append(movie_reviews.raw(fid))
        labels.append(cat)

# 2) 학습/검증 분리
X_train, X_test, y_train, y_test = train_test_split(
    docs, labels, test_size=0.2, random_state=42, stratify=labels
)

# 3) 파이프라인: TF-IDF → Linear SVM
pipe = make_pipeline(TfidfVectorizer(min_df=3, ngram_range=(1,2)),
                     LinearSVC())

# 4) 학습 & 평가
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

# 5) 사용 예시
samples = [
    "What a wonderful movie! Brilliant acting and a touching story.",
    "It was boring and way too long. I wouldn't recommend it."
]
print(pipe.predict(samples))  # ['pos', 'neg'] 예상


#### **예제3. 품사 태깅(POS Tagging)**
토큰 단위로 품사 라벨(POS tag) 를 예측하는 시퀀스 라벨링을, 간단한 맥락 기반 특징 + 로지스틱 회귀로 구현
- **태스크** : 품사 태깅
- **문제유형** : 시퀀스 라벨링(토큰 단위 분류)
- **데이터셋** : nltk, treebank
- **특징추출** : 주변 단어/점미사 등 수작업 특징
- **분류기** : Logistic Regression
- **핵심내용**: 지도학습의 핵심인 특징-->라벨) 체

In [ ]:
# --- 3. 품사 태깅: NLTK treebank + 간단 특징 + 로지스틱 회귀 ---
import nltk
nltk.download('treebank')
nltk.download('universal_tagset')

from nltk.corpus import treebank
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

# 1) (단어, 태그) 시퀀스 불러오기 (보편 태그셋 사용: NOUN, VERB, ADJ 등)
sents = treebank.tagged_sents(tagset='universal')

# 간단한 특징 함수: 현재/이전/다음 단어, 접미사/대문자/숫자 등
def token_features(sent_words, i):
    w = sent_words[i]
    prev_w = sent_words[i-1] if i-1 >= 0 else "<BOS>"
    next_w = sent_words[i+1] if i+1 < len(sent_words) else "<EOS>"
    feats = {
        "w.lower": w.lower(),
        "w.isupper": w.isupper(),
        "w.isdigit": w.isdigit(),
        "w.suffix2": w[-2:].lower(),
        "w.suffix3": w[-3:].lower(),
        "prev.lower": prev_w.lower(),
        "next.lower": next_w.lower(),
        "prev.isupper": prev_w.isupper() if isinstance(prev_w, str) else False,
        "next.isupper": next_w.isupper() if isinstance(next_w, str) else False,
    }
    return feats

# 2) 특징/라벨 벡터 만들기
X_dict, y = [], []
for sent in sents:
    words = [w for w, t in sent]
    tags  = [t for w, t in sent]
    for i in range(len(words)):
        X_dict.append(token_features(words, i))
        y.append(tags[i])

# 3) 훈련/테스트 분리
X_train_dict, X_test_dict, y_train, y_test = train_test_split(
    X_dict, y, test_size=0.2, random_state=42, stratify=y
)

# 4) Dict → 벡터화 → 로지스틱 회귀
vec = DictVectorizer(sparse=True)
X_train = vec.fit_transform(X_train_dict)
X_test  = vec.transform(X_test_dict)

clf = LogisticRegression(max_iter=300, n_jobs=None, multi_class='auto')
clf.fit(X_train, y_train)

# 5) 평가
pred = clf.predict(X_test)
print("Token-level Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))


### **[실습] 자연어 처리(NLP) 분야에서의 지도학습(회귀) 예**

#### **예제1 : 영화 리뷰 감정 점수 예측 (1-5점 척도)**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import re
from nltk.corpus import stopwords
import nltk

# 샘플 데이터 생성
reviews = [
    "This movie was absolutely amazing! Perfect story and acting.",
    "Great film with excellent cinematography and soundtrack.",
    "It was okay, nothing special but watchable.",
    "Not my favorite, but had some good moments.",
    "Terrible movie, waste of time and money.",
    "Outstanding performance by all actors!",
    "Average movie with predictable plot."
]

ratings = [5.0, 4.5, 3.0, 2.5, 1.0, 4.8, 3.2]

# 텍스트 전처리 함수
def preprocess_text(text):
    # 소문자 변환
    text = text.lower()
    # 특수문자 제거
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

# 전처리 적용
processed_reviews = [preprocess_text(review) for review in reviews]

# TF-IDF 벡터화
vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
X = vectorizer.fit_transform(processed_reviews)

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, ratings, test_size=0.3, random_state=42
)

# 선형 회귀 모델 훈련
model = LinearRegression()
model.fit(X_train, y_train)

# 예측 및 평가
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.3f}")
print(f"R² Score: {r2:.3f}")
print(f"실제값: {y_test}")
print(f"예측값: {y_pred}")

print()
상대오차율 = ((y_test - y_pred) / y_test) * 100
print(f"상대오차율: {상대오차율}%")

#### **예제2 : 텍스트 가독성 점수 예측**

In [ ]:
!pip install textstat

In [ ]:
import textstat
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

# 샘플 텍스트와 가독성 점수 (Flesch Reading Ease 기준)
texts = [
    "The cat sat on the mat.",
    "Artificial intelligence represents a paradigm shift in computational methodologies.",
    "Machine learning algorithms can process large amounts of data efficiently.",
    "Deep learning uses neural networks with multiple layers.",
    "This is a simple sentence that anyone can understand easily."
]

# 실제 Flesch Reading Ease 점수 계산
readability_scores = [textstat.flesch_reading_ease(text) for text in texts]

# 특징 추출 (문장 길이, 단어 수, 음절 수 등)
def extract_features(text):
    words = text.split()
    sentences = text.split('.')

    return [
        len(words),  # 단어 수
        len(sentences),  # 문장 수
        sum(len(word) for word in words) / len(words),  # 평균 단어 길이
        textstat.syllable_count(text),  # 음절 수
        len(text)  # 총 글자 수
    ]

# 특징 벡터 생성
features = [extract_features(text) for text in texts]
X = np.array(features)
y = np.array(readability_scores)

# 정규화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Random Forest 회귀 모델
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_scaled, y)

# 새로운 텍스트 예측
new_text = "Python is a programming language that is easy to learn and use."
new_features = scaler.transform([extract_features(new_text)])
predicted_score = rf_model.predict(new_features)[0]

print(f"새로운 텍스트: {new_text}")
print(f"예측된 가독성 점수: {predicted_score:.2f}")
print(f"실제 가독성 점수: {textstat.flesch_reading_ease(new_text):.2f}")

print()
actual = textstat.flesch_reading_ease(new_text)
상대오차율 = ((actual - predicted_score) / actual) * 100
print(f"상대오차율: {상대오차율:.2f}%")

#### **예제3 : 문서 유사도 점수 예측**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import SVR
import numpy as np

# 샘플 문서 쌍과 유사도 점수
doc_pairs = [
    ("The weather is nice today", "Today has beautiful weather"),
    ("I love programming", "Programming is my passion"),
    ("The cat is sleeping", "The dog is running"),
    ("Machine learning is fascinating", "AI and ML are interesting topics"),
    ("Python is great for data science", "Data analysis with Python is powerful")
]

# 실제 유사도 점수 (코사인 유사도 기반)
similarity_scores = []
for doc1, doc2 in doc_pairs:
    vectorizer = CountVectorizer().fit([doc1, doc2])
    vectors = vectorizer.transform([doc1, doc2])
    similarity = cosine_similarity(vectors[0], vectors[1])[0][0]
    similarity_scores.append(similarity)

# 특징 추출 (두 문서의 특징 결합)
def extract_pair_features(doc1, doc2):
    # 기본 통계
    len_diff = abs(len(doc1.split()) - len(doc2.split()))

    # 공통 단어 수
    words1 = set(doc1.lower().split())
    words2 = set(doc2.lower().split())
    common_words = len(words1.intersection(words2))

    # Jaccard 유사도
    jaccard_sim = len(words1.intersection(words2)) / len(words1.union(words2))

    return [len_diff, common_words, jaccard_sim]

# 특징 벡터 생성
features = [extract_pair_features(doc1, doc2) for doc1, doc2 in doc_pairs]
X = np.array(features)
y = np.array(similarity_scores)

# SVR 모델 훈련
svr_model = SVR(kernel='rbf', C=1.0, gamma='scale')
svr_model.fit(X, y)

# 예측 테스트
test_pair = ("Deep learning networks", "Neural networks for AI")
test_features = np.array([extract_pair_features(test_pair[0], test_pair[1])])
predicted_similarity = svr_model.predict(test_features)[0]

print(f"테스트 문서 쌍:")
print(f"문서 1: {test_pair[0]}")
print(f"문서 2: {test_pair[1]}")
print(f"예측된 유사도: {predicted_similarity:.3f}")

# 실제 코사인 유사도와 비교
vectorizer = CountVectorizer().fit(test_pair)
vectors = vectorizer.transform(test_pair)
actual_similarity = cosine_similarity(vectors[0], vectors[1])[0][0]
print(f"실제 코사인 유사도: {actual_similarity:.3f}")

print()
상대오차율 = ((actual_similarity - predicted_similarity) / actual_similarity) * 100
print(f"상대오차율: {상대오차율:.2f}%")



---



### **[실습] TF-IDF vs LLM 임베딩 감성 분류**
- 런타임 유형: GPU 선택

In [ ]:
!pip install -q sentence-transformers scikit-learn

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

# ============================================================
# 1. 데이터 불러오기
# ============================================================
# NSMC 영화 리뷰 데이터
# Naver Sentiment Movie Corpus
url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
df = pd.read_csv(url, sep="\t")

# 결측치 제거
df = df.dropna()

# 실습 시간을 줄이기 위해 3,000개 샘플만 사용
df = df.sample(3000, random_state=42)

# 입력 데이터 X, 정답 y
X = df["document"].tolist()
y = df["label"].values


# ============================================================
# 2. Train / Test 데이터 분리
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print("전체 데이터 :", len(X))
print("Train 데이터:", len(X_train))
print("Test 데이터 :", len(X_test))


# ============================================================
# 3. 방법 ① TF-IDF + Logistic Regression
# ============================================================
tfidf_model = make_pipeline(
    # 문자열을 TF-IDF 벡터로 변환
    TfidfVectorizer(
        analyzer="char_wb",      # 문자 단위 분석
        ngram_range=(2, 4),      # 2~4 글자 조합
        min_df=2                 # 최소 2개 문서에 등장
    ),

    # 분류 모델
    LogisticRegression(
        max_iter=2000
    )
)

# Train 데이터로 학습
tfidf_model.fit(X_train, y_train)

# Test 데이터 평가
tfidf_accuracy = tfidf_model.score(X_test, y_test)
print("TF-IDF Accuracy:", round(tfidf_accuracy, 3) )


# ============================================================
# 4. 방법 ② Sentence Embedding + Logistic Regression
# ============================================================
from sentence_transformers import SentenceTransformer

# 다국어 Sentence Transformer 모델
encoder = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    # "BAAI/bge-m3"
)

# ------------------------------------------------------------
# Train 문장 → Embedding Vector
# ------------------------------------------------------------
X_train_embedding = encoder.encode(
    X_train,
    normalize_embeddings=True,
    show_progress_bar=True
)

# ------------------------------------------------------------
# Test 문장 → Embedding Vector
# ------------------------------------------------------------
X_test_embedding = encoder.encode(
    X_test,
    normalize_embeddings=True,
    show_progress_bar=True
)
print("Train Embedding Shape:", X_train_embedding.shape)
print("Test Embedding Shape :", X_test_embedding.shape)


# ------------------------------------------------------------
# Embedding 벡터를 이용하여 Logistic Regression 학습
# ------------------------------------------------------------
embedding_model = LogisticRegression(max_iter=2000)
embedding_model.fit(X_train_embedding, y_train)

# Test 데이터 평가
embedding_accuracy = embedding_model.score(
    X_test_embedding,
    y_test
)
print("Embedding Accuracy:", round(embedding_accuracy, 3))


# ============================================================
# 5. 두 방법 성능 비교
# ============================================================
print("\n===== 모델 성능 비교 =====")
print(f"TF-IDF + Logistic Regression : {tfidf_accuracy:.3f}")
print(f"Embedding + Logistic Regression: {embedding_accuracy:.3f}")


- **Q3: TF-IDF 계수 상위 단어(문자)를 출력**

In [ ]:
import numpy as np

# Pipeline에서 TF-IDF와 Logistic Regression 꺼내기
vectorizer = tfidf_model.named_steps["tfidfvectorizer"]
classifier = tfidf_model.named_steps["logisticregression"]

# TF-IDF feature 이름
feature_names = vectorizer.get_feature_names_out()

# Logistic Regression이 학습한 계수
coef = classifier.coef_[0]

# 상위 10개
top_n = 10

# 긍정(1)에 영향을 많이 주는 feature
positive_idx = np.argsort(coef)[-top_n:][::-1]

# 부정(0)에 영향을 많이 주는 feature
negative_idx = np.argsort(coef)[:top_n]

print("===== 긍정에 영향을 많이 주는 TF-IDF Feature =====")
for idx in positive_idx:
    print(f"{feature_names[idx]:10s} {coef[idx]:.4f}")

print("\n===== 부정에 영향을 많이 주는 TF-IDF Feature =====")
for idx in negative_idx:
    print(f"{feature_names[idx]:10s} {coef[idx]:.4f}")



---



## **2. AI 모델링 프로세스**

### **[실습] 학생 성적데이터로 합격 여부 예측**

In [ ]:
from random import seed
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

data = {  # 02차시 [실습 1]과 동일한 '문제 있는' 학생 데이터
    "study_hours": [1, 2, 3, 4, np.nan, 6, 7, 8, 2, 3, 4, 5, 6, 7, 100, 4],
    "attendance":  [55, 60, 65, 70, 75, 80, 85, 95, 58, np.nan, 72, 76, 82, 88, 90, 70],
    "assignment":  [40, 50, 55, 60, 65, 75, 80, 95, 45, 55, 68, 72, 82, 87, 90, 60],
    "major": ["CS","CS","AI","SW","AI","CS","SW","AI","CS",None,"SW","AI","CS","SW","AI","SW"],
    "pass": [0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0],
}
df = pd.DataFrame(data)
df.loc[(df["study_hours"] < 0) | (df["study_hours"] > 24), "study_hours"] = np.nan  # 도메인 규칙
df = df.drop_duplicates()                                                            # 분리 전 중복 제거
X, y = df.drop(columns=["pass"]), df["pass"]

num = ["study_hours", "attendance", "assignment"]
cat = ["major"]
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore")), cat),
])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf = Pipeline([("pre", pre), ("model", LogisticRegression(max_iter=1000))])
clf.fit(Xtr, ytr)                       # ← 전처리 fit + 모델 fit이 Train에서만 일어난다
print("✅ Accuracy:", clf.score(Xte, yte))
print("✅ 학습된 parameter(w):", clf.named_steps["model"].coef_.round(2))
print("✅ hyperparameter:", clf.named_steps["model"].get_params()["C"])


- **Baseline — 복잡한 모델은 최소한 Dummy보다 좋아야 한다**

In [ ]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
print("[실습1] Dummy Accuracy:", dummy.score(Xte, yte), "| Pipeline Accuracy:", clf.score(Xte, yte))


### [실습] LLM이 짠 파이프라인 검수하기

- Gemini API 연결 : https://aistudio.google.com/api-keys

In [ ]:
# 런타임 다시 시작(Restart Runtime) 후 아래 셀만 실행하세요.
!pip install -q -U google-genai scikit-learn scipy

In [ ]:
import sklearn
import scipy

print(f"scikit-learn version: {sklearn.__version__}")
print(f"scipy version: {scipy.__version__}")

In [ ]:
from google import genai
from google.colab import userdata

# Colab Secrets에서 Gemini API Key 가져오기
api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini API 연결 완료")


- Gemini에게 머신러닝 코드 생성 요청

In [ ]:
gen_model = "gemini-3.5-flash"

spec = """
[문제]
학생 중도이탈 예측 (이진 분류)

[데이터]
- 출석률
- 과제제출수
- 중간고사점수
- BMI
- 학년
- 데이터 크기: 1,000행

[제약]
- 결측치 5%
- 클래스 불균형 9:1
- scikit-learn 사용

[요구]
다음 조건을 만족하는 Python 코드를 작성하라.

1. Pipeline + ColumnTransformer를 사용한다.
2. train/test 데이터를 먼저 분리한다.
3. 전처리기는 Train 데이터에만 fit한다.
4. train_test_split에서 stratify를 사용한다.
5. 클래스 불균형 처리를 위해 class_weight를 사용한다.
6. 평가 지표로 F1-score와 PR-AUC를 사용한다.
7. 최종 Test 데이터는 마지막에 한 번만 평가한다.
8. 데이터 누수가 발생하지 않도록 한다.
9. 실행 가능한 scikit-learn 코드만 출력한다.
"""

response = client.models.generate_content(
    model=gen_model,
    contents=spec
)

print(response.text)

- 응답에서 Python 코드만 추출

In [ ]:
import re

code_block_match = re.search(r"```python\n(.*?)```", response.text, re.DOTALL)

if code_block_match:
    code = code_block_match.group(1).strip()
else:
    # Fallback if no markdown block is found, try to strip common prefixes/suffixes
    code = response.text.strip()
    if code.startswith("```python"):
        code = code[len("```python"):]
    elif code.startswith("```"):
        code = code[len("```"):]
    if code.endswith("```"):
        code = code[:-3]
    code = code.strip()

print("===== Gemini가 생성한 코드 =====")
print(code)

- 생성된 코드를 실행

In [ ]:
exec(code)

- 생성된 코드를 Gemini에게 다시 검토시키기

In [ ]:
review_prompt = f"""
다음 scikit-learn 코드를 검토하라.

특히 다음 사항을 확인하라.

1. 데이터 누수(Data Leakage) 가능성이 있는가?
2. Train / Validation / Test의 역할이 적절히 분리되어 있는가?
3. 클래스 불균형 분류에 적합한 평가 지표를 사용했는가?
4. 전처리가 Pipeline과 ColumnTransformer 내부에서 수행되는가?
5. 수정이 필요한 경우 수정된 전체 코드를 제시하라.
6. 각 수정 이유를 한 줄로 설명하라.

다음은 검토할 코드이다.

```python
{code}

"""

review = client.models.generate_content(
model=gen_model,
contents=review_prompt
)

print(review.text)



---



## **3. 학습데이터 분할 방법**

### 예제 : K-Fold 교차검증 (회귀)

In [ ]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 데이터 생성
# X, y = make_regression(n_samples=1000, n_features=10, noise=10, random_state=42)
# X, y = make_regression(
#     n_samples=200,      # 작은 데이터셋
#     n_features=15,      # 특징 수 증가
#     n_informative=8,    # 유용한 특징 수
#     noise=30,           # 높은 노이즈
#     random_state=42
# )
# # 더 큰 차이를 보이는 데이터
X, y = make_regression(
    n_samples=100,      # 매우 작은 데이터셋
    n_features=20,      # 특징 수 더 증가
    n_informative=5,    # 유용한 특징은 적게
    noise=50,           # 매우 높은 노이즈
    random_state=123    # 다른 시드값
)

def compare_evaluation_methods(X, y):
    """
    단일 분할 vs K-Fold 교차검증 간단 비교
    """
    print("=== 교차검증 유무 비교 ===")

    # 1. 단일 분할 (교차검증 X)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    single_r2 = r2_score(y_test, y_pred)

    # 2. K-Fold 교차검증 (교차검증 O)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(LinearRegression(), X, y, cv=kfold, scoring='r2')
    kfold_r2_mean = cv_scores.mean()
    kfold_r2_std = cv_scores.std()

    # 결과 출력
    print(f"단일 분할 R²: {single_r2:.3f}")
    print(f"K-Fold    R²: {kfold_r2_mean:.3f} ± {kfold_r2_std:.3f}")
    print(f"차이: {abs(single_r2 - kfold_r2_mean):.3f}")

    if kfold_r2_std < 0.05:
        print("✅ K-Fold 결과가 안정적입니다.")
    else:
        print("⚠️ 성능 변동이 있습니다.")

# 실행
compare_evaluation_methods(X, y)

### [실습] 학습 데이터 분할 방법 테스트
- 앞의 **2.감성 분류 과정**에서 학습/테스트 데이터 분류를 5:5, 9:1로 했을 때 모델의 정확도가 변동이 있는지 확인해 보세요.
- **8:2** = Accuracy: **0.87**
- **5:5** = Accuracy: **?**
- **9:1** = Accuracy: **?**

In [ ]:
# --- 2. 감성 분석: NLTK movie_reviews (긍/부정) ---
import nltk
nltk.download('movie_reviews')

from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, accuracy_score

# 1) 문서/라벨 구성
docs = []
labels = []
for cat in movie_reviews.categories():          # 'pos', 'neg'
    for fid in movie_reviews.fileids(cat):
        docs.append(movie_reviews.raw(fid))
        labels.append(cat)


- **조건1: 학습:테스트=5:5**

In [ ]:
# 2) 학습/검증 분리
X_train, X_test, y_train, y_test = train_test_split(
    docs, labels, test_size=0.5, random_state=42, stratify=labels
)

# 3) 파이프라인: TF-IDF → Linear SVM
pipe = make_pipeline(TfidfVectorizer(min_df=3, ngram_range=(1,2)),
                     LinearSVC())

# 4) 학습 & 평가
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

# 5) 사용 예시
samples = [
    "What a wonderful movie! Brilliant acting and a touching story.",
    "It was boring and way too long. I wouldn't recommend it."
]
print(pipe.predict(samples))  # ['pos', 'neg'] 예상

### **[실습] 데이터 분할 방법에 따른 특징 확인**
- 앞에서 사용한 학생 성적데이터 그대로 사용
- 참고: _03_ml_aihub_text_finance.ipynb 에서 자세히 확인

In [ ]:
import numpy as np
import pandas as pd

# =====================================================================
# [실습 0] 02차시 Pipeline 다시 실행 — 분류 (합격 여부)
# =====================================================================
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

data = {  # 02차시 [실습 1]과 동일한 '문제 있는' 학생 데이터
    "study_hours": [1, 2, 3, 4, np.nan, 6, 7, 8, 2, 3, 4, 5, 6, 7, 100, 4],
    "attendance":  [55, 60, 65, 70, 75, 80, 85, 95, 58, np.nan, 72, 76, 82, 88, 90, 70],
    "assignment":  [40, 50, 55, 60, 65, 75, 80, 95, 45, 55, 68, 72, 82, 87, 90, 60],
    "major": ["CS","CS","AI","SW","AI","CS","SW","AI","CS",None,"SW","AI","CS","SW","AI","SW"],
    "pass": [0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0],
}
df = pd.DataFrame(data)
df.loc[(df["study_hours"] < 0) | (df["study_hours"] > 24), "study_hours"] = np.nan  # 도메인 규칙
df = df.drop_duplicates()                                                            # 분리 전 중복 제거
X, y = df.drop(columns=["pass"]), df["pass"]

num = ["study_hours", "attendance", "assignment"]
cat = ["major"]
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore")), cat),
])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf = Pipeline([("pre", pre), ("model", LogisticRegression(max_iter=1000))])
clf.fit(Xtr, ytr)                       # ← 전처리 fit + 모델 fit이 Train에서만 일어난다
print("[실습0] Accuracy:", clf.score(Xte, yte))
print("[실습0] 학습된 parameter(w):", clf.named_steps["model"].coef_.round(2))
print("[실습0] hyperparameter:", clf.named_steps["model"].get_params()["C"])

# =====================================================================
# [실습 1] Baseline — 복잡한 모델은 최소한 Dummy보다 좋아야 한다
# =====================================================================
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
print("[실습1] Dummy Accuracy:", dummy.score(Xte, yte), "| Pipeline Accuracy:", clf.score(Xte, yte))

# =====================================================================
# [실습 2] 분할 비율과 난수(seed)의 영향 — 한 번의 점수를 믿어도 되는가?
# =====================================================================
from sklearn.datasets import load_breast_cancer
Xb, yb = load_breast_cancer(return_X_y=True)          # 569행, 30 feature, 이진분류
base = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
print("[실습2] test_size  mean   std  (seed 20개)")
for ratio in [0.1, 0.2, 0.3, 0.5]:
    scores = []
    for seed in range(20):
        a, b, c, d = train_test_split(Xb, yb, test_size=ratio, stratify=yb, random_state=seed)
        scores.append(base.fit(a, c).score(b, d))
    print(f"         {ratio:<9} {np.mean(scores):.3f}  {np.std(scores):.3f}")

# =====================================================================
# [실습 3] K-fold 교차검증 — 점수의 분포를 본다
# =====================================================================
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(base, Xb, yb, cv=cv, scoring="f1")
print("[실습3] 5-fold F1:", cv_scores.round(3), "mean=%.3f std=%.3f" % (cv_scores.mean(), cv_scores.std()))

# =====================================================================
# [실습 4] 데이터 누수(Leakage) 직접 만들어 보기 — Feature Selection 누수
# =====================================================================
from sklearn.feature_selection import SelectKBest, f_classif
rng = np.random.default_rng(0)
Xn = rng.normal(size=(100, 2000))              # 정보가 전혀 없는 잡음 2000개
yn = rng.integers(0, 2, size=100)             # 무작위 정답 → 진짜 정확도는 0.5
X_bad = SelectKBest(f_classif, k=20).fit_transform(Xn, yn)   # ✗ 전체 데이터에서 feature 선택
bad = cross_val_score(LogisticRegression(max_iter=2000), X_bad, yn, cv=5).mean()
good = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20),
                                     LogisticRegression(max_iter=2000)), Xn, yn, cv=5).mean()
print(f"[실습4] 누수 있음(가짜 성능): {bad:.3f}   올바른 Pipeline: {good:.3f}   (진짜 답: 0.5 근처)")




---



## **4.모델 평가 방법**

### 예제 : 회귀모델 평가지표

|지표 | 공식 | 특징  | 해석 |
|---|---|---|--|
|MSE  | 평균((실제-예측)²) | 이상치에 민감 | 작을수록 좋음 |
|MAE평균 | (|실제-예측|) | 이상치에 강건 | 작을수록 좋음|
|R² | 1 - (오차분산/전체분산) | 설명력 측정 | 1에 가까울수록 좋음 |

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 샘플 데이터 생성
X, y = make_regression(n_samples=100, n_features=5, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 훈련 및 예측
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("=== 회귀 모델 평가지표 3가지 ===")
print(f"실제값 예시: {y_test[:5]}")
print(f"예측값 예시: {y_pred[:5]}")

# 1. MSE (Mean Squared Error) - 평균 제곱 오차
mse = mean_squared_error(y_test, y_pred)
print(f"\n1️⃣ MSE: {mse:.2f}")
print("   의미: 오차의 제곱 평균 (이상치에 민감)")
print("   해석: 값이 작을수록 좋음")

# 2. MAE (Mean Absolute Error) - 평균 절대 오차
mae = mean_absolute_error(y_test, y_pred)
print(f"\n2️⃣ MAE: {mae:.2f}")
print("   의미: 오차의 절댓값 평균 (이상치에 강건)")
print("   해석: 값이 작을수록 좋음, 직관적 이해 쉬움")

# 3. R² Score - 결정계수
r2 = r2_score(y_test, y_pred)
print(f"\n3️⃣ R² Score: {r2:.3f}")
print("   의미: 모델이 설명하는 분산 비율")
print("   해석: 0~1, 1에 가까울수록 좋음")

# 간단한 성능 평가
if r2 > 0.8:
    print("\n✅ 우수한 모델 성능")
elif r2 > 0.6:
    print("\n✅ 양호한 모델 성능")
else:
    print("\n⚠️ 모델 개선 필요")

### **[Tip] 모델의 파라미터 확인하는 방법**

In [ ]:
# 모델의 파라미터 확인하는 방법
print(model.get_params().keys())  # 파라미터 이름들
print(model.get_params())         # 파라미터 전체와 값

### 예제 : 분류모델 평가지표

|지표 | 공식 | 언제 중요한가 | 해석|
|---|---|---|--|
|Accuracy | (TP+TN)/(TP+TN+FP+FN) | 클래스가 균형잡힌 경우 | 1에 가까울수록 좋음|
|Precision | TP/(TP+FP) | 오탐(False Positive)이 치명적일 때 | 1에 가까울수록 좋음|
|Recall | TP/(TP+FN) | 놓침(False Negative)이 치명적일 때 | 1에 가까울수록 좋음|
|F1-Score | 2×(Precision×Recall)/(Precision+Recall) | 균형잡힌 평가가 필요할 때 | 1에 가까울수록 좋음|

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 샘플 분류 데이터 생성
X, y = make_classification(n_samples=1000, n_features=10, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 훈련 및 예측
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("=== 분류 모델 평가지표 4가지 ===")
print(f"실제값 예시: {y_test[:10]}")
print(f"예측값 예시: {y_pred[:10]}")

# 1. Accuracy (정확도)
accuracy = accuracy_score(y_test, y_pred)
print(f"\n1️⃣ Accuracy: {accuracy:.3f}")
print("   의미: 전체 중 맞게 예측한 비율")
print("   해석: 1에 가까울수록 좋음")

# 2. Precision (정밀도)
precision = precision_score(y_test, y_pred)
print(f"\n2️⃣ Precision: {precision:.3f}")
print("   의미: 양성으로 예측한 것 중 실제 양성인 비율")
print("   해석: 거짓 양성(오탐)을 줄이고 싶을 때 중요")

# 3. Recall (재현율)
recall = recall_score(y_test, y_pred)
print(f"\n3️⃣ Recall: {recall:.3f}")
print("   의미: 실제 양성 중 올바르게 예측한 비율")
print("   해석: 거짓 음성(놓침)을 줄이고 싶을 때 중요")

# 4. F1-Score (F1 점수)
f1 = f1_score(y_test, y_pred)
print(f"\n4️⃣ F1-Score: {f1:.3f}")
print("   의미: Precision과 Recall의 조화평균")
print("   해석: 균형잡힌 성능 평가")

# 혼동행렬 (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred)
print(f"\n📊 혼동행렬:")
print(f"   실제\\예측   0    1")
print(f"        0    {cm[0,0]:3}  {cm[0,1]:3}")
print(f"        1    {cm[1,0]:3}  {cm[1,1]:3}")

# 간단한 성능 평가
print(f"\n=== 성능 평가 ===")
if accuracy > 0.9:
    print("✅ 매우 우수한 성능")
elif accuracy > 0.8:
    print("✅ 우수한 성능")
elif accuracy > 0.7:
    print("✅ 양호한 성능")
else:
    print("⚠️ 성능 개선 필요")

# 정밀도 vs 재현율 트레이드오프
if precision > recall:
    print("📈 정밀도가 높음: 오탐이 적음")
else:
    print("📈 재현율이 높음: 놓치는 것이 적음")

In [ ]:
# 모델의 파라미터 확인하는 방법
print(model.get_params().keys())  # 파라미터 이름들
print(model.get_params())         # 파라미터 전체와 값

### **[실습] LLM 출력은 무엇으로 평가하나**

LLM의 생성 결과를 전통적인 자동평가와 LLM 기반 평가로 비교해보는 실습
> - BLEU/ROUGE는 “정답과 얼마나 비슷하게 썼는가?”를 보고,
> - LLM-as-a-Judge는 “내용이 실제로 얼마나 좋은가?”를 판단
- BLEU와 ROUGE는 기본적으로 문장의 의미 자체보다는 표면에 나타난 단어의 일치 정도를 계산 --> 표면 일치 지표

In [ ]:
# ==========================================================
# 1. 라이브러리 설치
# ==========================================================
!pip install -q google-genai evaluate rouge_score sacrebleu

In [ ]:
# ==========================================================
# 2. 라이브러리
# ==========================================================
import evaluate

from google import genai
from google.colab import userdata


# ==========================================================
# 3. Gemini API 연결
# ==========================================================
api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)


In [ ]:
# ==========================================================
# 4. Reference와 질문
# ==========================================================
ref = "기계학습은 데이터로부터 규칙을 스스로 찾아내는 기술이다."
question = "기계학습을 한 문장으로 정의해줘."


# ==========================================================
# 5. Gemini 답변 생성
# ==========================================================
gen_model = "gemini-3.6-flash"
response = client.models.generate_content(
    model=gen_model,
    contents=question
)
pred = response.text.strip()


print("=" * 60)
print("Reference")
print("=" * 60)
print(ref)

print()

print("=" * 60)
print("Gemini Prediction")
print("=" * 60)
print(pred)


In [ ]:
# ==========================================================
# 6. ROUGE 평가
# ==========================================================
rouge_metric = evaluate.load("rouge")
rouge = rouge_metric.compute(
    predictions=[pred],
    references=[ref]
)


# ==========================================================
# 7. BLEU 평가
# ==========================================================
bleu_metric = evaluate.load("sacrebleu")
bleu = bleu_metric.compute(
    predictions=[pred],
    references=[[ref]]
)


print()
print("=" * 60)
print("자동 평가 결과")
print("=" * 60)

print("ROUGE-1 :", round(rouge["rouge1"], 3))
print("ROUGE-2 :", round(rouge["rouge2"], 3))
print("ROUGE-L :", round(rouge["rougeL"], 3))
print("BLEU    :", round(bleu["score"], 2))

In [ ]:
# ==========================================================
# 8. LLM-as-a-Judge
# ==========================================================
judge_prompt = f"""
당신은 기계학습 답변을 평가하는 평가자입니다.

[질문]
{question}

[정답]
{ref}

[모델 답변]
{pred}

다음 기준으로 모델 답변을 각각 5점 만점으로 평가하세요.

1. 정확성 (accuracy)
2. 관련성 (relevance)
3. 명확성 (clarity)

각 점수에 대한 이유를 간단히 설명하세요.

다음 JSON 형식으로만 출력하세요.

{{
    "accuracy": 점수,
    "relevance": 점수,
    "clarity": 점수,
    "reason": "평가 이유"
}}
"""

judge = client.models.generate_content(
    model=gen_model,
    contents=judge_prompt
)


print()
print("=" * 60)
print("LLM-as-a-Judge 평가")
print("=" * 60)

print(judge.text)



---

